# Spatial TX registration to slab

1. Run st2block to generate affine transforms between st and blockface images. The outputs of these affines are specified in `transforms_path`
2. Obtain block2slab transforms  
    a. In this case, it was performed with SVG
3. Create dict of slab_id to slab_image



In [1]:
from hmba_stx_registration import get_metadata_df, Specimen, process_barcode
from hmba_stx_registration.st_to_slab import get_bf_affines_from_svg
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime
import logging
import os

os.makedirs("logs", exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    handlers=[
        logging.FileHandler("logs/st_to_slab_registration.log"),
        logging.StreamHandler(),
    ],
)

# date=datetime.now().strftime("%Y%m%d")
mapping_date = "20260528"
output_date = "20260610"
base_path = Path("/home/mike/workspace/data/macaque_brain/QM24.50.002")
sptx_path = base_path / "sptx"
barcodes_path = sptx_path / "xenium"
transforms_path = sptx_path / "st2block_output_20260610"
um_per_px = 20
mm_per_px = um_per_px / 1000
metadata_df = get_metadata_df(barcodes_path)
slab_ids = range(41, 54)
table_label = "subclass_label"

/home/mike/workspace/data/macaque_brain/QM24.50.002/sptx/xenium/blockface_images/blockface_images_specimen_metadata.json does not exist, skipping
/home/mike/workspace/data/macaque_brain/QM24.50.002/sptx/xenium/QM24.50.002_section_metadata.csv is not a directory, skipping
/home/mike/workspace/data/macaque_brain/QM24.50.002/sptx/xenium/registration_plots/registration_plots_specimen_metadata.json does not exist, skipping
/home/mike/workspace/data/macaque_brain/QM24.50.002/sptx/xenium/QM24.50.002_section_metadata_20260305.csv is not a directory, skipping
/home/mike/workspace/data/macaque_brain/QM24.50.002/sptx/xenium/barcode_csv/barcode_csv_specimen_metadata.json does not exist, skipping


In [2]:
bf_affines = {}
slab_imgs = {}
for slab_id in slab_ids:
    slab_imgs[slab_id] = plt.imread(base_path / "slabs" / "combined_stack" / f"CX{slab_id}_a_frozen.png")
    svg_path = base_path / f"QM24.50.002.CX.{slab_id}.svg"
    if not svg_path.exists():
        print(f"Missing SVG: {svg_path}")
        continue
    bf_affines_slab = get_bf_affines_from_svg(svg_path, base_path)
    bf_affines.update(bf_affines_slab)

# save bf_affines to json
import json
with open(base_path / f"bf_affines_{output_date}.json", "w") as f:
    json.dump({k: v.tolist() for k, v in bf_affines.items()}, f)

In [29]:
barcodes = metadata_df['barcode'].tolist()
for barcode in barcodes:
    specimen = Specimen(barcode, barcodes_path, date=mapping_date)
    if specimen.slab_name not in ['QM24.50.002.CX.53']:
        continue
    process_barcode(
        specimen,
        bf_affines,
        slab_imgs,
        transforms_path,
        table_label,
        um_per_px,
        mapping_date,
        output_date,
        version="0.3",
        sync_to_s3=True,
        sync_dryrun=False,
    )


INFO:hmba_stx_registration.st_to_slab:[1496653085] Copied block QC → QM24.50.002.CX.53.06.01.02_registration_block_qc_20260610.png
INFO:hmba_stx_registration.st_to_slab:[1496653085] Saved transform manifest: QM24.50.002.CX.53.06.01.02_coarse_transform_to_slab_mm_20260610.json
INFO:hmba_stx_registration.st_to_slab:[1496653085] Saved run manifest: QM24.50.002.CX.53.06.01.02_run_manifest_20260610.json
INFO:hmba_stx_registration.utils:Running: aws s3 sync /home/mike/workspace/data/macaque_brain/QM24.50.002/sptx/xenium/1496653085/registration_results_20260610 s3://hmba-macaque-wg-802451596237-us-west-2/hmba_aim_2/Xenium/QM24.50.002/1496653085/registration_results_20260610 --profile storage --only-show-errors
INFO:hmba_stx_registration.st_to_slab:[1496653085] Synced to S3
INFO:hmba_stx_registration.st_to_slab:[1496653085] Done (QM24.50.002.CX.53.06.01.02)
INFO:hmba_stx_registration.st_to_slab:[1496599670] Copied block QC → QM24.50.002.CX.53.05.01.02_registration_block_qc_20260610.png
INFO:hm

In [13]:
from hmba_stx_registration import apply_slab_transforms, plot_single_slab_scatter
import pandas as pd

barcodes_df = pd.read_csv(barcodes_path / "barcode_csv/QM24.50.002_section_metadata.csv", index_col=0)
figs_path = barcodes_path / "registration_plots" / f"registration_results_{output_date}"
figs_path.mkdir(parents=True, exist_ok=True)

exclude = ["401 Oligo_NN"]

for (slab, plane), group in barcodes_df.groupby(["slab", "slab_plane"], sort=True):
    if slab not in [51, 52, 53]:
        continue
    bc_list = group["barcode"].tolist()
    print(f"Processing slab {slab} / plane {plane} with {sorted(group.block.to_list())} blocks")
    df = apply_slab_transforms(barcodes_path, registration_results_date=output_date, barcodes=bc_list, mapping_date=mapping_date)
    if df.empty:
        print(f"Slab {slab} / {plane}: no data, skipping")
        continue
    fig = plot_single_slab_scatter(
        df,
        label_col="neighborhood_label",
        exclude_labels=set(exclude),
        title=f"Slab {slab} – plane {plane}",
    )
    fig.savefig(figs_path / f"QM24.50.002.CX.{int(slab)}.{plane}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Slab {slab} / {plane}: {len(df)} cells, {len(bc_list)} barcodes")

Processing slab 51 / plane Plane01 with [1, 2, 3, 4, 5, 7, 8, 9] blocks


INFO:hmba_stx_registration:[1434318461] Appended 117931 rows
INFO:hmba_stx_registration:[1495164772] Appended 188830 rows
INFO:hmba_stx_registration:[1493578411] Appended 151636 rows
INFO:hmba_stx_registration:[1435186227] Appended 189191 rows
INFO:hmba_stx_registration:[1491895140] Appended 159961 rows
INFO:hmba_stx_registration:[1493588248] Appended 169348 rows
INFO:hmba_stx_registration:[1491900371] Appended 154185 rows
INFO:hmba_stx_registration:[1491897629] Appended 163227 rows


Slab 51 / Plane01: 1294309 cells, 8 barcodes
Processing slab 51 / plane Plane02 with [1] blocks


INFO:hmba_stx_registration:[1434319422] Appended 103084 rows


Slab 51 / Plane02: 103084 cells, 1 barcodes
Processing slab 51 / plane Plane03 with [1, 2, 3, 4, 5, 6, 7, 8, 9] blocks


...

Slab 53 / Plane01: 1293948 cells, 7 barcodes
Processing slab 53 / plane Plane03 with [1, 2, 3, 4, 5, 6, 7] blocks


INFO:hmba_stx_registration:[1496664465] Appended 237340 rows
INFO:hmba_stx_registration:[1496599793] Appended 252047 rows
INFO:hmba_stx_registration:[1496451301] Appended 129777 rows
INFO:hmba_stx_registration:[1496455335] Appended 346115 rows
INFO:hmba_stx_registration:[1496653193] Appended 187462 rows
INFO:hmba_stx_registration:[1496594881] Appended 131244 rows
INFO:hmba_stx_registration:[1434006749] Appended 157134 rows


Slab 53 / Plane03: 1441119 cells, 7 barcodes
Processing slab 53 / plane Plane05 with [1, 2, 3, 5, 6, 7] blocks


INFO:hmba_stx_registration:[1496653332] Appended 214010 rows
INFO:hmba_stx_registration:[1496451438] Appended 160583 rows
INFO:hmba_stx_registration:[1496664683] Appended 193701 rows
INFO:hmba_stx_registration:[1496455608] Appended 148160 rows
INFO:hmba_stx_registration:[1434006726] Appended 187840 rows
INFO:hmba_stx_registration:[1496599944] Appended 243908 rows


Slab 53 / Plane05: 1148202 cells, 6 barcodes


In [3]:
from hmba_stx_registration.utils import sync_dir_to_s3

figs_path = barcodes_path / "registration_plots"
sync_dir_to_s3(
    local_dir=figs_path,
    bucket="hmba-macaque-wg-802451596237-us-west-2",
    s3_dir=f"hmba_aim_2/Xenium/QM24.50.002/registration_plots",
    profile="storage",
    dryrun=False,
    delete=False,
)

INFO:hmba_stx_registration.utils:Running: aws s3 sync /home/mike/workspace/data/macaque_brain/QM24.50.002/sptx/xenium/registration_plots s3://hmba-macaque-wg-802451596237-us-west-2/hmba_aim_2/Xenium/QM24.50.002/registration_plots --profile storage --only-show-errors


In [15]:
from hmba_stx_registration import upload_plots_to_s3
from pathlib import Path
figs_path = barcodes_path / "registration_plots/registration_results_20260610"
png_files = figs_path.glob("*.png")
upload_plots_to_s3(
    png_paths=png_files,
    bucket="hmba-macaque-wg-802451596237-us-west-2",
    s3_dir="hmba_aim_2/Xenium/QM24.50.002/registration_plots/registration_results_20260610",
    profile="storage",
    dryrun=False,
)

INFO:hmba_stx_registration.utils:Uploading QM24.50.002.CX.51.Plane03.png → s3://hmba-macaque-wg-802451596237-us-west-2/hmba_aim_2/Xenium/QM24.50.002/registration_plots/registration_results_20260610/QM24.50.002.CX.51.Plane03.png
INFO:hmba_stx_registration.utils:Uploading QM24.50.002.CX.51.Plane01.png → s3://hmba-macaque-wg-802451596237-us-west-2/hmba_aim_2/Xenium/QM24.50.002/registration_plots/registration_results_20260610/QM24.50.002.CX.51.Plane01.png
INFO:hmba_stx_registration.utils:Uploading QM24.50.002.CX.52.Plane05.png → s3://hmba-macaque-wg-802451596237-us-west-2/hmba_aim_2/Xenium/QM24.50.002/registration_plots/registration_results_20260610/QM24.50.002.CX.52.Plane05.png
INFO:hmba_stx_registration.utils:Uploading QM24.50.002.CX.52.Plane03.png → s3://hmba-macaque-wg-802451596237-us-west-2/hmba_aim_2/Xenium/QM24.50.002/registration_plots/registration_results_20260610/QM24.50.002.CX.52.Plane03.png
INFO:hmba_stx_registration.utils:Uploading QM24.50.002.CX.51.Plane02.png → s3://hmba-mac

['s3://hmba-macaque-wg-802451596237-us-west-2/hmba_aim_2/Xenium/QM24.50.002/registration_plots/registration_results_20260610/QM24.50.002.CX.51.Plane03.png',
 's3://hmba-macaque-wg-802451596237-us-west-2/hmba_aim_2/Xenium/QM24.50.002/registration_plots/registration_results_20260610/QM24.50.002.CX.51.Plane01.png',
 's3://hmba-macaque-wg-802451596237-us-west-2/hmba_aim_2/Xenium/QM24.50.002/registration_plots/registration_results_20260610/QM24.50.002.CX.52.Plane05.png',
 's3://hmba-macaque-wg-802451596237-us-west-2/hmba_aim_2/Xenium/QM24.50.002/registration_plots/registration_results_20260610/QM24.50.002.CX.52.Plane03.png',
 's3://hmba-macaque-wg-802451596237-us-west-2/hmba_aim_2/Xenium/QM24.50.002/registration_plots/registration_results_20260610/QM24.50.002.CX.51.Plane02.png',
 's3://hmba-macaque-wg-802451596237-us-west-2/hmba_aim_2/Xenium/QM24.50.002/registration_plots/registration_results_20260610/QM24.50.002.CX.51.Plane07.png',
 's3://hmba-macaque-wg-802451596237-us-west-2/hmba_aim_2/X

In [ ]:
# Debug a specific slab/plane — QM24.50.002.CX.49 / Plane01
import importlib, hmba_stx_registration
importlib.reload(hmba_stx_registration)
from hmba_stx_registration import apply_slab_transforms, plot_single_slab_scatter

target_slab, target_plane = 49, "Plane01"
group = barcodes_df[(barcodes_df["slab"] == target_slab) & (barcodes_df["slab_plane"] == target_plane)]
print(f"Barcodes for slab {target_slab} / {target_plane}: {len(group)}")
print(group["barcode"].tolist())

bc_list = group["barcode"].tolist()
df = apply_slab_transforms(barcodes_path, registration_results_date=date, barcodes=bc_list)
print(f"Cells loaded: {len(df)}")

if not df.empty:
    fig = plot_single_slab_scatter(
        df,
        exclude_labels={"Oligodendrocyte", "Vascular"},
        title=f"Slab {target_slab} – plane {target_plane}",
    )
    fig.show()

